# Generative Adversarial Network (GAN) (2-27-26)

* Last of the unsupervised learning algorithms we will cover this week.
* Note that I will be phrasing the below notes in terms of image generations with is the most common use of GANs but they can also be use on audio, text (with some difficulty), and other types of data.
* [Original Paper](https://arxiv.org/abs/1406.2661)

## Part 1: Generator
* Takes in a vector of random noise (usually Gaussian or Uniform) and produces a fake image.
* Made of transposed CNN layers which upscale
    * Increase spatial dimensions instead of decreasing for feature extration

## Part 2: Discriminator
* Takes in a sample image and produces a binary output, is that image real or fake (1 is a real sample, 0 is a fake sample).
* Made of CNN layers which downscale

## Adversarial Relationship
* Discriminator wants to always produce a 0 when fed an image from the Generator.
* Generator wants a 1 produced when its generated images are fed to the Discriminator. 
* Simultaneously attempting to minimize the value function for the discriminator but maximize the value function for the generator.

## Algorithm
1. Sample real images
2. Sample noise from a given distribution
3. Generate a fake batch of images using the sampled noise
4. Update the discriminator using both th real and the fake images
5. Sample new noise from the distribution
6. Generate a new set of images from this noise
7. Feed the new set of images to the discriminator and get its predictions
8. Update the generator based on the predictions


## Problems with GANs
* Mode Collapse: Generator can learn to map many input values to the same output
* Vanishing Gradients
* Non-Convergence of the Optimizer (need to balance minimization for the generator and maximization for the discriminator)

## Types of GANs
* Deep Concolutional Generative Ad versarial Network (DCGAN) (implemented below)
    * [Paper](https://arxiv.org/abs/1511.06434)
    * Uses convolutional layers with no pooling, batch normalization, and a ReLU on the Generator and LeakyReLU on the Discriminator. 
    * Tends to produce stable image generator


## Comparison to Other Models
* GANs produce sharper images than VAEs but VAEs are much easier and faster to train.
* Diffusion models produce similar quality images (if not better quality) images compared to GANs and are much easier to train
    * GANs use to be the best model for image generation but it has moved over to diffusion models


## Pytorch Implementation

In [2]:
#############
## IMPORTS ##
#############
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image

In [3]:
######################
## DEVICE SELECTION ##
######################
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
###############
## VARIABLES ##
###############
# Just gathering all the variables in one place for easy access and modification.

# Location of the data
data_root = "./data"
# Location to save generated samples
out_dir = "./samples"
# Number of epochs to train
epochs = 20
# Size of the batches during training
batch_size = 128
# Learning rate for optimizer
lr = 2e-4
# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5
# Latent dimension
nz = 100
# Feature maps in Generator
ngf = 64
# Feature maps in Discriminator
ndf = 64

In [5]:
############################
## MAKE OUTPUT DIRECTORY ##
###########################
os.makedirs(out_dir, exist_ok=True)

In [6]:
#####################
## IMPORT THE DATA ##
#####################
# We need to transform the data into tensors and normalize it to be in the range [-1, 1] for better training of the GAN.
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),  # maps [0,1] -> [-1,1]
])

# Import the MNIST dataset using torchvision, applying the transformations defined above.
dataset = torchvision.datasets.MNIST(root=data_root, train=True, download=True, transform=transform)

# Create a DataLoader to handle batching and shuffling of the dataset during training.
loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)

In [7]:
###############
## GENERATOR ##
###############
class Generator(nn.Module):
    """
    The Generator takes a random noise vector z and transforms it into a fake image.
    """
    def __init__(self, nz=100, ngf=64, nc=1):
        """
        Inputs:
            nz: Dimension of the input noise vector.
            ngf: Number of generator feature maps.
            nc: Number of output channels (1 for grayscale images).
        Returns:
            None.
        Initializes the Generator model with a series of ConvTranspose2d layers to upsample the noise vector into a 28x28 image.
        """
        # Call the parent class constructor
        super().__init__()
        # Define the network architecture using nn.Sequential for simplicity.
        self.net = nn.Sequential(
            # Transformation: 1x1 -> 7x7
            nn.ConvTranspose2d(nz, ngf * 4, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # Transformation: 7x7 -> 14x14
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # Transformation: 14x14 -> 28x28
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # refine channels, keep 28x28
            nn.Conv2d(ngf, nc, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        """
        Inputs:
            z: A batch of noise vectors with shape (N, nz, 1, 1).
        Returns:
            A batch of generated images with shape (N, nc, 28, 28).
        Performs a forward pass through the network, transforming the input noise vector into a generated image.
        """
        return self.net(z)

In [8]:
###################
## DISCRIMINATOR ##
###################

class Discriminator(nn.Module):
    """
    The Discriminator takes an image x and outputs a logit indicating whether the image is real or fake.
    """
    def __init__(self, ndf=64, nc=1):
        """
        Inputs:
            ndf: Number of discriminator feature maps.
            nc: Number of input channels (1 for grayscale images).
        Returns:
            None.
        Initializes the Discriminator model with a series of Conv2d layers to downsample the input image and 
        produce a logit indicating real/fake. The critical fix is to ensure that the output is always a single 
        scalar per image, regardless of the input image size.
        """
        # Call the parent class constructor
        super().__init__()
        # Define the feature extraction layers using nn.Sequential for simplicity.
        self.features = nn.Sequential(
            # 28x28 -> 14x14
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 14x14 -> 7x7
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 7x7 -> 4x4  (stride 2 with padding keeps it stable)
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # produces a 1-channel logit map (could be spatial)
        self.logit_head = nn.Conv2d(ndf * 4, 1, kernel_size=3, stride=1, padding=1, bias=False)

        # Define a pooling layer to ensure we get a single logit per image, regardless of the spatial dimensions of the feature map.
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        """
        Inputs:
            x: A batch of images with shape (N, nc, 28, 28).
        Returns:
            A batch of logits with shape (N,).
        Performs a forward pass through the network, transforming the input image into a logit indicating real/fake.
        """
        x = self.features(x)         # (N, C, H, W)
        x = self.logit_head(x)       # (N, 1, H, W)
        x = self.pool(x)             # (N, 1, 1, 1) 
        return x.view(-1)            # (N,)

In [9]:
###############################
## INITIALIZATION OF WEIGHTS ##
###############################
def weights_init(m):
    """
    Inputs:
        m: A module in the network.
    Returns:
        None.
    Initializes the weights of the network using a normal distribution for convolutional layers and batch normalization layers, 
    following the DCGAN paper's recommendations for stable training.
    """
    # Get the class name of the module to determine how to initialize its weights.
    name = m.__class__.__name__
    # Initialize convolutional layers with a normal distribution centered at 0 with a standard deviation of 0.02.
    if "Conv" in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    # Initialize batch normalization layers with a normal distribution centered at 1 with a standard deviation of 0.02 
    # for weights, and set biases to 0.
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [10]:
############################################
## DEFINE THE GENERATOR AND DISCRIMINATOR ##
############################################
# Define the Generator and Discriminator models, move them to the appropriate device (CPU or GPU), and apply the weight 
# initialization function to ensure stable training.
G = Generator(nz, ngf).to(device)
D = Discriminator(ndf).to(device)
G.apply(weights_init)
D.apply(weights_init)


Discriminator(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (logit_head): Conv2d(256, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (pool): AdaptiveAvgPool2d(output_size=1)
)

In [ ]:
#############################################
## DEFINE THE LOSS FUNCTION AND OPTIMIZERS ##
#############################################
# Define the loss function and optimizers. For the optimizer, we use Adam with the learning rate and beta1 parameter defined earlier. There
# is one optimizer for the generator and one for the discriminator. The betas are the coefficients used for computing running averages of gradient 
# and its square in the Adam optimizer. This is commonly used in GANs to stabilize training.
criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
optD = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))


In [12]:
##################
## RANDOM NOISE ##
##################
# Create a fixed noise vector for generating consistent samples during training visualization. This allows us to see how the 
# Generator's output evolves over time for the same input noise.
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

In [13]:
###################
## TRAINING LOOP ##
###################
# Keep track of training steps
step = 0
# For each epoch
for epoch in range(1, epochs + 1):
    # For each batch load just the x data so we are doing unsupervised learning again
    for real, _ in loader:
        # Move real images to device
        real = real.to(device)
        # Get batch size
        bsz = real.size(0)

        ## TRAIN DISCRIMINATOR 
        D.zero_grad(set_to_none=True)
        # Get discriminator logits for real images
        logits_real = D(real)                     # (bsz,)
        # Create labels for real images
        labels_real = torch.ones_like(logits_real)
        # Compute discriminator loss on real images
        lossD_real = criterion(logits_real, labels_real)

        # Generate fake images
        noise = torch.randn(bsz, nz, 1, 1, device=device)
        fake = G(noise).detach()
        # Get discriminator logits for fake images
        logits_fake = D(fake)                     # (bsz,)
        # Create labels for fake images
        labels_fake = torch.zeros_like(logits_fake)
        # Compute discriminator loss on fake images
        lossD_fake = criterion(logits_fake, labels_fake)

        # Combine losses and update discriminator
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        ## TRAIN GENERATOR
        G.zero_grad(set_to_none=True)
    
        # Generate fake images
        noise = torch.randn(bsz, nz, 1, 1, device=device)
        fake = G(noise)
        # Get discriminator logits for fake images
        logits = D(fake)                          # (bsz,)
        # Create labels for generator (want D(fake) -> real)
        labels_gen = torch.ones_like(logits)      
        # Compute generator loss
        lossG = criterion(logits, labels_gen)

        lossG.backward()
        optG.step()
        # Print training progress
        if step % 200 == 0:
            print("Epoch:",  epoch, "/", epochs, "] Step", step, 
                  "lossD =", lossD.item(), "lossG=", lossG.item())
        step += 1

    # Save sample grid each epoch. We use the fixed noise defined in the above cell here.
    G.eval()
    with torch.no_grad():
        samples = G(fixed_noise).cpu()
    G.train()

    grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    out_path = os.path.join(out_dir, f"mnist_dcgan_epoch_{epoch:03d}.png")
    save_image(grid, out_path)
    print(f"Saved samples: {out_path}")

# This took 85 minutes to run on my Mac but note that it also produces an image with each epoch
# which does take time. You can comment out this block of code if you want to speed up the training
# or you can reduce the number of epochs to 10 or 5 for testing purposes.

Epoch: 1 / 20 ] Step 0 lossD = 1.3875771760940552 lossG= 0.8073136806488037
Epoch: 1 / 20 ] Step 200 lossD = 1.0698004961013794 lossG= 1.2605818510055542
Epoch: 1 / 20 ] Step 400 lossD = 0.8874969482421875 lossG= 1.0611069202423096
Saved samples: ./samples/mnist_dcgan_epoch_001.png
Epoch: 2 / 20 ] Step 600 lossD = 1.0556094646453857 lossG= 1.5004687309265137
Epoch: 2 / 20 ] Step 800 lossD = 0.9368589520454407 lossG= 0.9843684434890747
Saved samples: ./samples/mnist_dcgan_epoch_002.png
Epoch: 3 / 20 ] Step 1000 lossD = 0.9452411532402039 lossG= 1.1131501197814941
Epoch: 3 / 20 ] Step 1200 lossD = 1.003309965133667 lossG= 0.689931333065033
Epoch: 3 / 20 ] Step 1400 lossD = 1.146701455116272 lossG= 0.5028172731399536
Saved samples: ./samples/mnist_dcgan_epoch_003.png
Epoch: 4 / 20 ] Step 1600 lossD = 1.0240771770477295 lossG= 1.2296640872955322
Epoch: 4 / 20 ] Step 1800 lossD = 0.9322392344474792 lossG= 1.2080174684524536
Saved samples: ./samples/mnist_dcgan_epoch_004.png
Epoch: 5 / 20 ] 